In [1]:
import os

In [2]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\MLoPs\\Movie_recommendation_system\\Notebook'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\MLoPs\\Movie_recommendation_system'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    movies_data_path: Path
    credits_data_path: Path
    transformed_data_path: Path


In [6]:
from Movie_Recommendation_system.constants import *
from  Movie_Recommendation_system.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            movies_data_path = config.movies_data_path,
            credits_data_path = config.credits_data_path,
            transformed_data_path = config.transformed_data_path
        )

        return data_transformation_config

In [8]:
from Movie_Recommendation_system import logger
from sklearn.model_selection import train_test_split
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer

In [9]:
import pandas as pd
import ast
from pathlib import Path

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    # 🔹 helper: extract names from JSON-like string
    def _convert(self, obj):
        try:
            return [i["name"] for i in ast.literal_eval(obj)]
        except Exception:
            return []

    # 🔹 helper: extract director name
    def _fetch_director(self, obj):
        try:
            for i in ast.literal_eval(obj):
                if i.get("job") == "Director":
                    return i.get("name")
        except Exception:
            pass
        return ""

    def transform_data(self):
        # 1️⃣ load datasets
        movies = pd.read_csv(self.config.movies_data_path)
        credits = pd.read_csv(self.config.credits_data_path)

        credits = credits.drop(columns=["title"])

        logger.info('Dataset Loaded.....')

        # 2️⃣ merge (correct stage)
        movies = movies.merge(
            credits,
            left_on="id",
            right_on="movie_id"
        )

        logger.info('Merge two files......')

        # 3️⃣ select required columns
        movies = movies[
            ["id", "title", "overview", "genres", "keywords", "cast", "crew"]
        ]

        logger.info('selected columns....')

        # 4️⃣ drop missing rows
        movies.dropna(inplace=True)

        # 5️⃣ feature extraction
        movies["genres"] = movies["genres"].apply(self._convert)
        movies["keywords"] = movies["keywords"].apply(self._convert)
        movies["cast"] = movies["cast"].apply(lambda x: self._convert(x)[:3])
        movies["director"] = movies["crew"].apply(self._fetch_director)

        logger.info('Feature Extraction Done....')

        # 6️⃣ create tags column
        movies["tags"] = (
            movies["overview"]
            + " "
            + movies["genres"].apply(lambda x: " ".join(x))
            + " "
            + movies["keywords"].apply(lambda x: " ".join(x))
            + " "
            + movies["cast"].apply(lambda x: " ".join(x))
            + " "
            + movies["director"]
        )

        logger.info('Create tags Columns')

        # 7️⃣ final clean
        movies["tags"] = movies["tags"].str.lower()
        final_df = movies[["id", "title", "tags"]]

        # 8️⃣ save transformed data
        final_df.to_csv(
            self.config.transformed_data_path,
            index=False
        )

        logger.info('Save transformed Data in Artifacts')


In [10]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()

    data_transformation = DataTransformation(data_transformation_config)
    data_transformation.transform_data()

except Exception as e:
    raise e


[2026-02-10 13:03:29,336: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-02-10 13:03:29,341: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-10 13:03:29,357: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-02-10 13:03:29,360: INFO: common: created directory at: artifacts]
[2026-02-10 13:03:29,363: INFO: common: created directory at: artifacts/data_transformation]
[2026-02-10 13:03:30,095: INFO: 3665219823: Dataset Loaded.....]
[2026-02-10 13:03:30,108: INFO: 3665219823: Merge two files......]
[2026-02-10 13:03:30,117: INFO: 3665219823: selected columns....]
[2026-02-10 13:03:48,563: INFO: 3665219823: Feature Extraction Done....]
[2026-02-10 13:03:48,617: INFO: 3665219823: Create tags Columns]
[2026-02-10 13:03:48,769: INFO: 3665219823: Save transformed Data in Artifacts]
